# 03 - Generative VAE: Pretrain, Finetune, and Visualization (NaN/Inf Fix Applied)

**Note**: This notebook includes automated NaN/Inf detection and sanitization before pretraining to prevent batch skips.

# 03 - Generative VAE: Pretrain, Finetune, and Visualization

This notebook demonstrates a lightweight two-phase VAE workflow: a short pretraining pass (ZINC-style), followed by a finetune pass on target-like data. For demo purposes this runs on synthetic data and produces an RDKit grid image of example molecules.

In [18]:
%pip install pandas numpy matplotlib seaborn scikit-learn torch-geometric rdkit-pypi tensorboard biopython

# Setup and imports
# Resolve the project root up front so the pretraining and fine-tuning phases can reuse the same data and model paths.
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
import numpy as np
from core.vae_architecture import GraphVAE

# These helpers keep the notebook runnable on both GPU and Apple Silicon, and normalize graph shapes before batching.
def get_working_device():
    if torch.cuda.is_available():
        try:
            torch.zeros(1).to('cuda')
            return torch.device('cuda')
        except Exception:
            pass
    if torch.backends.mps.is_available():
        try:
            torch.zeros(1).to('mps')
            return torch.device('mps')
        except Exception:
            pass
    return torch.device('cpu')

def normalize_edge_index(data):
    edge_index = getattr(data, 'edge_index', None)
    if edge_index is None:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if not hasattr(edge_index, 'ndim') or edge_index.ndim != 2:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if 0 in edge_index.shape:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if edge_index.shape[0] == 2:
        return data
    if edge_index.shape[1] == 2:
        data.edge_index = edge_index.t().contiguous()
        return data
    data.edge_index = torch.zeros((2, 0), dtype=torch.long)
    return data

def truncate_graph_edges(data, max_nodes):
    edge_index = getattr(data, 'edge_index', None)
    edge_attr = getattr(data, 'edge_attr', None)
    if edge_index is None or not hasattr(edge_index, 'ndim') or edge_index.ndim != 2 or edge_index.shape[1] == 0:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        if edge_attr is not None:
            data.edge_attr = edge_attr.new_zeros((0, edge_attr.shape[-1]))
        return data
    if edge_index.shape[0] == 2:
        node_mask = (edge_index[0] < max_nodes) & (edge_index[1] < max_nodes)
    else:
        node_mask = (edge_index[:, 0] < max_nodes) & (edge_index[:, 1] < max_nodes)
        edge_index = edge_index.t().contiguous()
    edge_index = edge_index[:, node_mask]
    if edge_attr is not None and hasattr(edge_attr, 'shape') and edge_attr.shape[0] == node_mask.shape[0]:
        edge_attr = edge_attr[node_mask]
    if edge_index.numel() == 0:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        if edge_attr is not None:
            data.edge_attr = edge_attr.new_zeros((0, edge_attr.shape[-1]))
        return data
    data.edge_index = edge_index
    if edge_attr is not None:
        data.edge_attr = edge_attr
    return data
OUT = ROOT / 'results' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
device = get_working_device()
print(f'Ready; device={device}')

Note: you may need to restart the kernel to use updated packages.
Ready; device=cuda


ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


In [29]:
# ===== PHASE 1: VAE PRETRAINING ON REAL ZINC DATA =====

DATA_PROC = ROOT / 'data' / 'processed'
DATA_RAW = ROOT / 'data' / 'raw'

print('='*60)
print('PHASE 1: Pre-training on ZINC (Real Large-Scale Data)')
print('='*60)

# Load SMILES from ZINC (either .smi or .csv)
zinc_smiles = []
zinc_file_smi = DATA_RAW / 'zinc250k.smi'
zinc_file_csv = DATA_RAW / 'zinc250k.csv'

if zinc_file_smi.exists():
    print(f'\nLoading ZINC from {zinc_file_smi.name}...')
    with open(zinc_file_smi, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                zinc_smiles.append(parts[0])
    print(f'  Loaded {len(zinc_smiles)} SMILES')
elif zinc_file_csv.exists():
    print(f'\nLoading ZINC from {zinc_file_csv.name}...')
    import pandas as pd
    zinc_df = pd.read_csv(zinc_file_csv)
    zinc_smiles = zinc_df['smiles'].astype(str).tolist()
    print(f'  Loaded {len(zinc_smiles)} SMILES')
else:
    print('ZINC file not found; using synthetic data fallback')
    zinc_smiles = None

if zinc_smiles:
    n_pretrain = min(200000, len(zinc_smiles))
    zinc_smiles = zinc_smiles[:n_pretrain]
    print(f'Using {len(zinc_smiles)} ZINC molecules for pretraining')
else:
    print('(Pretraining phase will skip real data)')

from rdkit import Chem
from rdkit.Chem import AllChem
from torch_geometric.data import Data as PyGData


def safe_float(val, default=0.0):
    try:
        out = float(val)
    except Exception:
        return default
    if not np.isfinite(out):
        return default
    return out


def get_node_features(atom):
    partial_charge = 0.0
    if atom.HasProp('_GasteigerCharge'):
        partial_charge = safe_float(atom.GetProp('_GasteigerCharge'), default=0.0)
    partial_charge = float(np.clip(partial_charge, -5.0, 5.0))

    return [
        float(atom.GetAtomicNum()),
        float(atom.GetTotalDegree()),
        float(atom.GetFormalCharge()),
        float(int(atom.GetHybridization())),
        float(int(atom.GetIsAromatic())),
        float(atom.GetMass()),
        float(atom.GetTotalNumHs()),
        partial_charge,
    ]


def get_edge_features(bond):
    bond_type = bond.GetBondType()
    return [
        int(bond_type == Chem.rdchem.BondType.SINGLE),
        int(bond_type == Chem.rdchem.BondType.DOUBLE),
        int(bond_type == Chem.rdchem.BondType.TRIPLE),
        int(bond_type == Chem.rdchem.BondType.AROMATIC),
    ]


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    try:
        AllChem.ComputeGasteigerCharges(mol)
    except Exception:
        pass

    x = [get_node_features(atom) for atom in mol.GetAtoms()]
    edge_index = [[], []]
    edge_attr = []

    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index[0].extend([a1, a2])
        edge_index[1].extend([a2, a1])
        features = get_edge_features(bond)
        edge_attr.extend([features, features])

    x_t = torch.tensor(x, dtype=torch.float)
    x_t = torch.nan_to_num(x_t, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)

    if len(edge_attr) == 0:
        edge_attr_t = torch.zeros((0, 4), dtype=torch.float)
        edge_index_t = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_attr_t = torch.tensor(edge_attr, dtype=torch.float)
        edge_attr_t = torch.nan_to_num(edge_attr_t, nan=0.0, posinf=1.0, neginf=0.0)
        edge_index_t = torch.tensor(edge_index, dtype=torch.long)

    data = PyGData(x=x_t, edge_index=edge_index_t, edge_attr=edge_attr_t)
    return data


def standardize_feature_tensor(features, eps=1e-6):
    if features is None or features.numel() == 0:
        return features
    feature_mean = features.mean(dim=0, keepdim=True)
    feature_std = features.std(dim=0, keepdim=True, unbiased=False).clamp_min(eps)
    standardized = (features - feature_mean) / feature_std
    return torch.nan_to_num(standardized, nan=0.0, posinf=0.0, neginf=0.0)


print('\nConverting ZINC SMILES to PyG graphs...')
zinc_graphs = []
if zinc_smiles:
    for i, smiles in enumerate(zinc_smiles):
        if i % 5000 == 0:
            print(f'  Processed {i}/{len(zinc_smiles)}')
        g = smiles_to_graph(smiles)
        if g is not None:
            zinc_graphs.append(g)

    zinc_graphs = [normalize_edge_index(g) for g in zinc_graphs]

    # Dataset-wide sanitization pass
    fixed_graphs = 0
    for g in zinc_graphs:
        changed = False
        if torch.isnan(g.x).any() or torch.isinf(g.x).any():
            g.x = torch.nan_to_num(g.x, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)
            changed = True
        if getattr(g, 'edge_attr', None) is not None and (torch.isnan(g.edge_attr).any() or torch.isinf(g.edge_attr).any()):
            g.edge_attr = torch.nan_to_num(g.edge_attr, nan=0.0, posinf=1.0, neginf=0.0)
            changed = True
        if changed:
            fixed_graphs += 1

    print(f'Converted {len(zinc_graphs)} valid ZINC graphs')
    print(f'Sanitization pass fixed {fixed_graphs} graphs')
else:
    print('(Skipping graph conversion: using synthetic fallback)')

edge_feat_dim = int(zinc_graphs[0].edge_attr.shape[1]) if zinc_graphs and getattr(zinc_graphs[0], 'edge_attr', None) is not None else 4
print(f'Inferred ZINC edge feature width: {edge_feat_dim}')

max_nodes = 50
epochs_pre = 15

vae = GraphVAE(
    node_features=8,
    edge_features=edge_feat_dim,
    hidden_dim=128,
    latent_dim=64,
    max_nodes=max_nodes,
    use_pocket_conditioning=False,
)
vae = vae.to(device)

optimizer = torch.optim.Adam(vae.parameters(), lr=5e-4)
criterion_recon = torch.nn.MSELoss()

print('\nTraining VAE pretraining phase')
print(f'  Graphs: {len(zinc_graphs)}, Epochs: {epochs_pre}, Max nodes: {max_nodes}')
print('  (Processing in mini-batches of 32 graphs)\n')

vae.train()

if len(zinc_graphs) > 0:
    batch_size = 32
    n_batches = (len(zinc_graphs) + batch_size - 1) // batch_size

    for ep in range(epochs_pre):
        total_loss = 0.0
        n_processed = 0
        skipped_batches = 0

        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(zinc_graphs))
            batch_graphs = zinc_graphs[start_idx:end_idx]

            if len(batch_graphs) == 0:
                continue

            optimizer.zero_grad()

            batch_x_list = []
            batch_edge_index_list = []
            batch_edge_attr_list = []
            batch_indices = []
            node_offset = 0

            for g_idx, g in enumerate(batch_graphs):
                x = g.x
                n_atoms = x.shape[0]

                if n_atoms < max_nodes:
                    x = torch.cat([x, torch.zeros(max_nodes - n_atoms, x.shape[1], device=x.device, dtype=x.dtype)], dim=0)
                else:
                    x = x[:max_nodes]

                x = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)
                batch_x_list.append(x)

                g = truncate_graph_edges(g, max_nodes)
                if g.edge_index.shape[1] > 0:
                    edge_idx = g.edge_index + node_offset
                    batch_edge_index_list.append(edge_idx)
                    ea = torch.nan_to_num(g.edge_attr, nan=0.0, posinf=1.0, neginf=0.0)
                    batch_edge_attr_list.append(ea)

                batch_indices.extend([g_idx] * max_nodes)
                node_offset += max_nodes

            X_batch = torch.cat(batch_x_list, dim=0)

            if batch_edge_index_list:
                edge_index_batch = torch.cat(batch_edge_index_list, dim=1)
                edge_attr_batch = torch.cat(batch_edge_attr_list, dim=0)
            else:
                edge_index_batch = torch.zeros((2, 0), dtype=torch.long)
                edge_attr_batch = torch.zeros((0, edge_feat_dim), dtype=torch.float)

            batch_tensor = torch.tensor(batch_indices, dtype=torch.long)
            global_feat = standardize_feature_tensor(torch.zeros(len(batch_graphs), 2))

            X_batch = X_batch.to(device)
            edge_index_batch = edge_index_batch.to(device)
            edge_attr_batch = edge_attr_batch.to(device)
            batch_tensor = batch_tensor.to(device)
            global_feat = global_feat.to(device)

            if not torch.isfinite(X_batch).all() or not torch.isfinite(edge_attr_batch).all():
                skipped_batches += 1
                print(f'Non-finite input tensors at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            mu, logvar = vae.encode(X_batch, edge_index_batch, edge_attr_batch, batch_tensor, global_feat)

            if not torch.isfinite(mu).all() or not torch.isfinite(logvar).all():
                skipped_batches += 1
                print(f'Non-finite encode output at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            mu = torch.clamp(mu, -30.0, 30.0)
            logvar = torch.clamp(logvar, -20.0, 20.0)

            z = vae.reparameterize(mu, logvar)
            node_logits, edge_adj, edge_type = vae.decode(z, pocket_embedding=None)

            if not torch.isfinite(node_logits).all() or not torch.isfinite(edge_adj).all() or not torch.isfinite(edge_type).all():
                skipped_batches += 1
                print(f'Non-finite decode output at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            X_reshaped = X_batch.view(len(batch_graphs), max_nodes, -1)
            loss_recon = criterion_recon(node_logits, X_reshaped)
            kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

            beta = 0.01 * (ep + 1) / epochs_pre
            loss = loss_recon + beta * kld

            if torch.isnan(loss) or not torch.isfinite(loss):
                skipped_batches += 1
                print(f'NaN/Inf loss at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(batch_graphs)
            n_processed += len(batch_graphs)

        avg_loss = total_loss / max(n_processed, 1)
        print(f'Epoch {ep+1}/{epochs_pre} - Loss: {avg_loss:.4f}, beta: {beta:.4f}, skipped: {skipped_batches}')
else:
    print('No ZINC graphs available; skipping pretraining')

models_dir = ROOT / 'models'
models_dir.mkdir(exist_ok=True, parents=True)
torch.save(vae.state_dict(), models_dir / 'zinc_pretrained_graphvae.pth')
print(f'\nPretrained VAE saved: {models_dir / "zinc_pretrained_graphvae.pth"}')

PHASE 1: Pre-training on ZINC (Real Large-Scale Data)

Loading ZINC from zinc250k.csv...
  Loaded 249455 SMILES
Using 200000 ZINC molecules for pretraining

Converting ZINC SMILES to PyG graphs...
  Processed 0/200000
  Processed 5000/200000
  Processed 10000/200000
  Processed 15000/200000
  Processed 20000/200000
  Processed 25000/200000
  Processed 30000/200000
  Processed 35000/200000
  Processed 40000/200000
  Processed 45000/200000
  Processed 50000/200000
  Processed 55000/200000
  Processed 60000/200000
  Processed 65000/200000
  Processed 70000/200000
  Processed 75000/200000
  Processed 80000/200000
  Processed 85000/200000
  Processed 90000/200000
  Processed 95000/200000
  Processed 100000/200000
  Processed 105000/200000
  Processed 110000/200000
  Processed 115000/200000
  Processed 120000/200000
  Processed 125000/200000
  Processed 130000/200000
  Processed 135000/200000
  Processed 140000/200000
  Processed 145000/200000
  Processed 150000/200000
  Processed 155000/200

In [35]:
# ===== PHASE 2: FINE-TUNING ON REAL EGFR DATA WITH POCKET CONDITIONING =====
print('\n' + '='*60)
print('PHASE 2: Fine-tuning on EGFR (With Pocket Conditioning)')
print('='*60)

import torch.nn.functional as F
from collections import Counter

device = get_working_device()
vae = vae.to(device)
print(f'Using device for fine-tuning: {device}')

# Load EGFR graphs if not already present in memory
if 'egfr_graphs' not in globals() or len(egfr_graphs) == 0:
    graph_file = DATA_PROC / 'graph_data.pt'
    if graph_file.exists():
        egfr_graphs = torch.load(graph_file)
        print(f'Loaded {len(egfr_graphs)} EGFR graphs from {graph_file.name}')
    else:
        egfr_graphs = []

if len(egfr_graphs) == 0:
    print('No EGFR graphs available; skipping fine-tuning.')
else:
    # Ensure edge index format is consistent
    egfr_graphs = [normalize_edge_index(g) for g in egfr_graphs]

    # Compute atom class distribution for CE weighting
    ATOM_SYMBOLS = ['C', 'N', 'O', 'F', 'S', 'Cl']
    class_counter = Counter()
    total_atoms = 0

    for g in egfr_graphs:
        if getattr(g, 'x', None) is None:
            continue
        atomic_nums = g.x[:, 0].to(torch.long).tolist()
        for at in atomic_nums:
            if at == 6:
                class_counter['C'] += 1
            elif at == 7:
                class_counter['N'] += 1
            elif at == 8:
                class_counter['O'] += 1
            elif at == 9:
                class_counter['F'] += 1
            elif at == 16:
                class_counter['S'] += 1
            elif at == 17:
                class_counter['Cl'] += 1
            else:
                class_counter['C'] += 1
            total_atoms += 1

    n_atom_classes = len(ATOM_SYMBOLS)
    class_counts = torch.tensor([class_counter.get(sym, 0) for sym in ATOM_SYMBOLS], dtype=torch.float)
    class_counts = class_counts.clamp_min(1.0)
    class_weights = (1.0 / class_counts)
    class_weights = class_weights / class_weights.sum() * n_atom_classes
    class_weights = class_weights.to(device)

    print(f'Atom class counts: {class_counts.tolist()}')
    print(f'Atom class weights: {class_weights.tolist()}')

    # Prepare real pocket embedding
    pocket_dim = getattr(vae.decoder, 'pocket_dim', 128)
    if 'pocket_embedding_tensor' in globals():
        real_pocket = pocket_embedding_tensor.to(device)
    elif 'real_pocket_array' in globals():
        real_pocket = torch.tensor(real_pocket_array, dtype=torch.float32, device=device).unsqueeze(0)
    else:
        from core.pocket_extractor import get_pocket_embedding
        pdb_file = DATA_RAW / 'pdb' / '3W2S.pdb'
        real_pocket_array = get_pocket_embedding(
            str(pdb_file),
            ligand_code='W2R',
            pocket_radius=7.0,
            embedding_dim=pocket_dim,
        )
        real_pocket = torch.tensor(real_pocket_array, dtype=torch.float32, device=device).unsqueeze(0)

    print(f'Using pocket embedding shape: {tuple(real_pocket.shape)}')

    # Freeze encoder, train decoder + atom classifier
    for p in vae.encoder.parameters():
        p.requires_grad = False

    if 'atom_classifier' not in globals() or atom_classifier.in_features != vae.decoder.node_features or atom_classifier.out_features != n_atom_classes:
        atom_classifier = torch.nn.Linear(vae.decoder.node_features, n_atom_classes).to(device)
    else:
        atom_classifier = atom_classifier.to(device)

    trainable_params = list(filter(lambda p: p.requires_grad, vae.parameters())) + list(atom_classifier.parameters())
    optimizer = torch.optim.Adam(trainable_params, lr=1e-3)
    criterion_recon = torch.nn.MSELoss()

    # Fine-tuning hyperparameters (Option B: Gumbel-Softmax ST)
    epochs_ft = 30
    batch_size = 16
    max_nodes = 50
    alpha_atom = 1.0
    lambda_halogen = 0.1
    lambda_edge = 0.01
    beta = 0.01
    tau_start = 1.0
    tau_end = 0.5
    tau_decay_epochs = 10

    # Determine expected edge feature width and halogen indices
    expected_edge_dim = int(getattr(vae, 'edge_features', (edge_feat_dim if 'edge_feat_dim' in globals() else 4)))
    halogen_idxs = [ATOM_SYMBOLS.index('F'), ATOM_SYMBOLS.index('Cl')]
    n_batches = (len(egfr_graphs) + batch_size - 1) // batch_size

    print(f'\nFine-tuning for {epochs_ft} epochs on {len(egfr_graphs)} EGFR graphs...')
    print(f'Expected edge feature dim: {expected_edge_dim}')

    vae.train()
    atom_classifier.train()

    for ep in range(epochs_ft):
        total_loss = 0.0
        total_ce = 0.0
        total_halogen_pen = 0.0
        total_edge_pen = 0.0
        n_processed = 0
        skipped_batches = 0

        tau_progress = min(ep + 1, tau_decay_epochs) / tau_decay_epochs
        tau = tau_start + (tau_end - tau_start) * tau_progress

        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(egfr_graphs))
            batch_graphs = egfr_graphs[start_idx:end_idx]

            if len(batch_graphs) == 0:
                continue

            optimizer.zero_grad()

            batch_x_list = []
            batch_edge_index_list = []
            batch_edge_attr_list = []
            batch_indices = []
            node_offset = 0

            for g_idx, g in enumerate(batch_graphs):
                g = truncate_graph_edges(g, max_nodes)
                x = g.x
                n_atoms = x.shape[0]

                if n_atoms < max_nodes:
                    x = torch.cat([x, torch.zeros(max_nodes - n_atoms, x.shape[1], device=x.device, dtype=x.dtype)], dim=0)
                else:
                    x = x[:max_nodes]

                x = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)
                batch_x_list.append(x)

                if getattr(g, 'edge_index', None) is not None and g.edge_index.shape[1] > 0:
                    edge_idx = g.edge_index + node_offset
                    batch_edge_index_list.append(edge_idx)

                    # Normalize edge_attr to expected width by padding/truncating
                    if getattr(g, 'edge_attr', None) is None or g.edge_attr.numel() == 0:
                        ea = torch.zeros((0, expected_edge_dim), dtype=torch.float)
                    else:
                        ea = torch.nan_to_num(g.edge_attr, nan=0.0, posinf=1.0, neginf=0.0)
                        if ea.dim() == 1:
                            ea = ea.unsqueeze(1)
                        cur = ea.shape[1]
                        if cur < expected_edge_dim:
                            pad = torch.zeros((ea.shape[0], expected_edge_dim - cur), device=ea.device, dtype=ea.dtype)
                            ea = torch.cat([ea, pad], dim=1)
                        elif cur > expected_edge_dim:
                            ea = ea[:, :expected_edge_dim]

                    batch_edge_attr_list.append(ea)

                batch_indices.extend([g_idx] * max_nodes)
                node_offset += max_nodes

            X_batch = torch.cat(batch_x_list, dim=0)

            if batch_edge_index_list:
                edge_index_batch = torch.cat(batch_edge_index_list, dim=1)
                edge_attr_batch = torch.cat(batch_edge_attr_list, dim=0)
            else:
                edge_index_batch = torch.zeros((2, 0), dtype=torch.long)
                edge_attr_batch = torch.zeros((0, expected_edge_dim), dtype=torch.float)

            batch_tensor = torch.tensor(batch_indices, dtype=torch.long)
            global_feat = standardize_feature_tensor(torch.zeros(len(batch_graphs), 2))
            batch_pockets = real_pocket.repeat(len(batch_graphs), 1)

            X_batch = X_batch.to(device)
            edge_index_batch = edge_index_batch.to(device)
            edge_attr_batch = edge_attr_batch.to(device)
            batch_tensor = batch_tensor.to(device)
            global_feat = global_feat.to(device)
            batch_pockets = batch_pockets.to(device)

            if not torch.isfinite(X_batch).all() or not torch.isfinite(edge_attr_batch).all():
                skipped_batches += 1
                continue

            mu, logvar = vae.encode(X_batch, edge_index_batch, edge_attr_batch, batch_tensor, global_feat)
            if not torch.isfinite(mu).all() or not torch.isfinite(logvar).all():
                skipped_batches += 1
                continue

            mu = torch.clamp(mu, -30.0, 30.0)
            logvar = torch.clamp(logvar, -20.0, 20.0)

            z = vae.reparameterize(mu, logvar)
            node_logits, edge_adj, edge_type = vae.decode(z, pocket_embedding=batch_pockets)

            if not torch.isfinite(node_logits).all() or not torch.isfinite(edge_adj).all() or not torch.isfinite(edge_type).all():
                skipped_batches += 1
                continue

            X_reshaped = X_batch.view(len(batch_graphs), max_nodes, -1)
            loss_recon = criterion_recon(node_logits, X_reshaped)
            kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

            # Atom CE supervision (label from atomic number in feature channel 0)
            atom_class_logits = atom_classifier(node_logits)
            atomic_nums = X_reshaped[..., 0].long().clamp_min(0)
            atom_labels = torch.zeros_like(atomic_nums)
            atom_labels[atomic_nums == 6] = 0
            atom_labels[atomic_nums == 7] = 1
            atom_labels[atomic_nums == 8] = 2
            atom_labels[atomic_nums == 9] = 3
            atom_labels[atomic_nums == 16] = 4
            atom_labels[atomic_nums == 17] = 5

            valid_mask = atomic_nums > 0
            if valid_mask.any():
                ce_loss = F.cross_entropy(
                    atom_class_logits[valid_mask],
                    atom_labels[valid_mask],
                    weight=class_weights,
                )
            else:
                ce_loss = torch.tensor(0.0, device=device)

            # Gumbel-Softmax straight-through over atom classes
            gumbel_atom = F.gumbel_softmax(atom_class_logits, tau=tau, hard=True, dim=-1)

            # Halogen penalty based on expected halogen count per graph
            halogen_prob = gumbel_atom[..., halogen_idxs].sum(dim=-1)
            halogen_penalty = F.relu(halogen_prob.sum(dim=-1) - 1.0).mean()

            # Edge sparsity penalty
            edge_prob_mean = torch.sigmoid(edge_adj).mean()
            edge_penalty = edge_prob_mean

            loss = (
                loss_recon
                + beta * kld
                + alpha_atom * ce_loss
                + lambda_halogen * halogen_penalty
                + lambda_edge * edge_penalty
            )

            if torch.isnan(loss) or not torch.isfinite(loss):
                skipped_batches += 1
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()

            bsz = len(batch_graphs)
            total_loss += float(loss.item()) * bsz
            total_ce += float(ce_loss.item()) * bsz
            total_halogen_pen += float(halogen_penalty.item()) * bsz
            total_edge_pen += float(edge_penalty.item()) * bsz
            n_processed += bsz

        avg_loss = total_loss / max(n_processed, 1)
        avg_ce = total_ce / max(n_processed, 1)
        avg_hal = total_halogen_pen / max(n_processed, 1)
        avg_edge = total_edge_pen / max(n_processed, 1)

        print(
            f'Epoch {ep+1:02d}/{epochs_ft} - '
            f'loss={avg_loss:.4f}, ce={avg_ce:.4f}, hal={avg_hal:.4f}, edge={avg_edge:.4f}, '
            f'tau={tau:.3f}, skipped={skipped_batches}'
        )

    # Save fine-tuned weights
    models_dir = ROOT / 'models'
    models_dir.mkdir(exist_ok=True, parents=True)
    torch.save(vae.state_dict(), models_dir / 'egfr_finetuned_graphvae.pth')
    print(f'\nFine-tuned VAE saved: {models_dir / "egfr_finetuned_graphvae.pth"}')


PHASE 2: Fine-tuning on EGFR (With Pocket Conditioning)
Using device for fine-tuning: cuda
Atom class counts: [487738.0, 108754.0, 46507.0, 12355.0, 4581.0, 7785.0]
Atom class weights: [0.026717737317085266, 0.11982323229312897, 0.28019988536834717, 1.0547353029251099, 2.8446311950683594, 1.6738927364349365]
Using pocket embedding shape: (1, 128)

Fine-tuning for 30 epochs on 19476 EGFR graphs...
Expected edge feature dim: 4
Epoch 01/30 - loss=5.8693, ce=1.9226, hal=2.6078, edge=0.0121, tau=0.950, skipped=0
Epoch 02/30 - loss=5.6047, ce=1.8942, hal=2.3302, edge=0.0001, tau=0.900, skipped=0
Epoch 03/30 - loss=5.5063, ce=1.8687, hal=2.3040, edge=0.0001, tau=0.850, skipped=0
Epoch 04/30 - loss=5.4407, ce=1.8532, hal=2.2666, edge=0.0000, tau=0.800, skipped=0
Epoch 05/30 - loss=5.3946, ce=1.8394, hal=2.2298, edge=0.0000, tau=0.750, skipped=0
Epoch 06/30 - loss=5.3622, ce=1.8313, hal=2.2124, edge=0.0000, tau=0.700, skipped=0
Epoch 07/30 - loss=5.3339, ce=1.8258, hal=2.1932, edge=0.0000, tau

In [36]:
# ===== PHASE 3: GENERATIVE SAMPLING =====

print('\n' + '='*60)
print('PHASE 3: Molecule Generation from Learned Latent Space')
print('='*60)

from torch_geometric.data import Data as PyGData
from rdkit import Chem

vae.eval()
n_samples = 1000
print(f'\nGenerating {n_samples} molecules by sampling latent space...\n')

latent_dim = vae.latent_dim
pocket_dim = getattr(vae.decoder, 'pocket_dim', 128)
thresholds_to_try = [0.3, 0.4, 0.5]
node_keep_fraction = 0.25
atom_temperature = 0.7
max_halogens_before_penalty = 0
chlorine_penalty = 3.0
report_every = 100
baseline_samples = min(25, n_samples)

# Use the real extracted T790M pocket if available; otherwise recompute it from the PDB.
if 'pocket_embedding_tensor' in globals():
    real_pocket = pocket_embedding_tensor.to(device)
elif 'real_pocket_array' in globals():
    real_pocket = torch.tensor(real_pocket_array, dtype=torch.float32, device=device).unsqueeze(0)
else:
    from core.pocket_extractor import get_pocket_embedding
    pdb_file = DATA_RAW / 'pdb' / '3W2S.pdb'
    real_pocket_array = get_pocket_embedding(
        str(pdb_file),
        ligand_code='W2R',
        pocket_radius=7.0,
        embedding_dim=pocket_dim,
    )
    real_pocket = torch.tensor(real_pocket_array, dtype=torch.float32, device=device).unsqueeze(0)

print(f'✓ Using real pocket embedding with shape: {tuple(real_pocket.shape)}')

# Optional global features placeholder for downstream packaging/scoring.
global_feature_template = torch.zeros(1, 20, device=device)

ATOM_SYMBOLS = ['C', 'N', 'O', 'F', 'S', 'Cl']
MAX_VALENCE = {
    'C': 4,
    'N': 3,
    'O': 2,
    'F': 1,
    'S': 6,
    'Cl': 1,
}
BOND_ORDER_BY_IDX = {
    0: 1.0,
    1: 2.0,
    2: 3.0,
    3: 1.5,
}
BOND_TYPE_BY_ORDER = {
    1.0: Chem.rdchem.BondType.SINGLE,
    1.5: Chem.rdchem.BondType.AROMATIC,
    2.0: Chem.rdchem.BondType.DOUBLE,
    3.0: Chem.rdchem.BondType.TRIPLE,
}
HALOGENS = {'F', 'Cl'}


def infer_atom_symbol(node_features, halogen_count=0):
    atom_logits = torch.as_tensor(node_features[: len(ATOM_SYMBOLS)], dtype=torch.float32)
    if atom_logits.numel() == 0:
        return 'C'

    if halogen_count >= max_halogens_before_penalty:
        atom_logits[ATOM_SYMBOLS.index('Cl')] -= chlorine_penalty
        atom_logits[ATOM_SYMBOLS.index('F')] -= 1.5

    scaled_logits = atom_logits / max(atom_temperature, 1e-6)
    probabilities = torch.softmax(scaled_logits, dim=0)
    sampled_idx = int(torch.multinomial(probabilities, num_samples=1).item())
    return ATOM_SYMBOLS[min(sampled_idx, len(ATOM_SYMBOLS) - 1)]


def atom_order_for_symbol(symbol):
    return MAX_VALENCE.get(symbol, 4)


def bond_order_from_logits(bond_logits):
    return BOND_ORDER_BY_IDX.get(int(torch.argmax(bond_logits).item()), 1.0)


def bond_order_to_type(bond_order):
    return BOND_TYPE_BY_ORDER.get(bond_order, Chem.rdchem.BondType.SINGLE)


def build_sparse_graph(node_tensor, edge_adj_tensor, edge_type_tensor, threshold):
    """Convert decoder outputs into sparse PyG/RDKit outputs."""
    node_scores = node_tensor.abs().sum(dim=-1)
    target_nodes = max(2, min(node_tensor.size(0), int(round(node_tensor.size(0) * node_keep_fraction))))
    active_indices = torch.topk(node_scores, k=target_nodes).indices.sort().values

    active_node_features = node_tensor[active_indices].detach().cpu()
    active_adj_probs = torch.sigmoid(edge_adj_tensor)[active_indices][:, active_indices]
    active_edge_types = edge_type_tensor[active_indices][:, active_indices]

    candidate_edges = []
    n_active = active_indices.numel()
    for i in range(n_active):
        for j in range(i + 1, n_active):
            prob = float(active_adj_probs[i, j].item())
            candidate_edges.append((prob, i, j))

    # Keep a sparse graph even when probabilities are flat around 0.5.
    candidate_edges.sort(reverse=True, key=lambda item: item[0])
    max_bonds = max(1, min(len(candidate_edges), n_active * 2))
    selected_edges = [edge for edge in candidate_edges if edge[0] >= threshold][:max_bonds]
    if not selected_edges:
        selected_edges = candidate_edges[:max_bonds]

    edge_index_pairs = []
    edge_attr_pairs = []
    bond_type_dim = active_edge_types.size(-1)
    for _, i, j in selected_edges:
        bond_logits = active_edge_types[i, j]
        bond_type_idx = int(torch.argmax(bond_logits).item())
        edge_index_pairs.extend([[i, j], [j, i]])
        one_hot = [1.0 if k == bond_type_idx else 0.0 for k in range(bond_type_dim)]
        edge_attr_pairs.extend([one_hot, one_hot])

    if edge_index_pairs:
        edge_index = torch.tensor(edge_index_pairs, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr_pairs, dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, bond_type_dim), dtype=torch.float32)

    pyg_graph = PyGData(
        x=active_node_features,
        edge_index=edge_index,
        edge_attr=edge_attr,
        global_features=global_feature_template.cpu().clone(),
        pocket_embedding=real_pocket.cpu().clone(),
    )

    return pyg_graph, active_indices, selected_edges, active_node_features, active_edge_types


def build_rdkit_smiles(node_features_cpu, selected_edges, edge_type_cpu, fallback_single_bonds=False):
    """Reconstruct an RDKit molecule with valence-aware bond placement."""
    mol = Chem.RWMol()
    atom_symbols = []
    halogen_count = 0

    for node_features in node_features_cpu.numpy():
        symbol = infer_atom_symbol(node_features, halogen_count=halogen_count)
        if symbol in HALOGENS:
            halogen_count += 1
        atom = Chem.Atom(symbol)
        atom.SetFormalCharge(0)
        atom.SetNoImplicit(False)
        atom.SetNumExplicitHs(0)
        atom_symbols.append(symbol)
        mol.AddAtom(atom)

    current_valence = [0.0 for _ in atom_symbols]

    # Add strongest bonds first.
    sorted_edges = sorted(selected_edges, key=lambda item: item[0], reverse=True)
    for _, i, j in sorted_edges:
        bond_logits = edge_type_cpu[i, j]
        preferred_order = bond_order_from_logits(bond_logits)
        bond_order_candidates = [preferred_order, 1.0] if not fallback_single_bonds else [1.0]

        for bond_order in bond_order_candidates:
            max_i = atom_order_for_symbol(atom_symbols[i])
            max_j = atom_order_for_symbol(atom_symbols[j])
            if current_valence[i] + bond_order <= max_i and current_valence[j] + bond_order <= max_j:
                try:
                    mol.AddBond(int(i), int(j), bond_order_to_type(bond_order))
                    current_valence[i] += bond_order
                    current_valence[j] += bond_order
                    break
                except Exception:
                    continue

    rdkit_mol = mol.GetMol()
    try:
        rdkit_mol.UpdatePropertyCache(strict=False)
    except Exception:
        pass

    try:
        Chem.SanitizeMol(rdkit_mol)
        smiles = Chem.MolToSmiles(rdkit_mol, canonical=True)
        return rdkit_mol, smiles
    except Exception:
        # Soft fallback: rebuild the same scaffold with single bonds only.
        try:
            fallback_mol = Chem.RWMol()
            for symbol in atom_symbols:
                atom = Chem.Atom(symbol)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                fallback_mol.AddAtom(atom)

            fallback_valence = [0.0 for _ in atom_symbols]
            for _, i, j in sorted_edges:
                if fallback_valence[i] + 1.0 <= atom_order_for_symbol(atom_symbols[i]) and fallback_valence[j] + 1.0 <= atom_order_for_symbol(atom_symbols[j]):
                    try:
                        fallback_mol.AddBond(int(i), int(j), Chem.rdchem.BondType.SINGLE)
                        fallback_valence[i] += 1.0
                        fallback_valence[j] += 1.0
                    except Exception:
                        pass

            fallback_rdkit_mol = fallback_mol.GetMol()
            try:
                fallback_rdkit_mol.UpdatePropertyCache(strict=False)
            except Exception:
                pass
            Chem.SanitizeMol(fallback_rdkit_mol)
            smiles = Chem.MolToSmiles(fallback_rdkit_mol, canonical=True)
            return fallback_rdkit_mol, smiles
        except Exception:
            return None, None

sample_logs = []
generated_graphs = []
filtered_smiles = []


def passes_quick_halogen_filter(smiles, max_total_halogens=1):
    if not smiles:
        return False
    if 'Cl' in smiles:
        return False
    halogen_tokens = ['Cl', 'Br', 'F', 'I']
    total_halogens = sum(smiles.count(token) for token in halogen_tokens)
    return total_halogens <= max_total_halogens


with torch.no_grad():
    for i in range(n_samples):
        z_sample = torch.randn(1, latent_dim, device=device)
        node_features, edge_adj_logits, edge_type_logits = vae.decode(z_sample, pocket_embedding=real_pocket)
        
        threshold_results = []
        best_pack = None
        best_threshold = None
        for threshold in thresholds_to_try:
            pyg_graph, active_indices, selected_edges, active_nodes_cpu, active_edge_types_cpu = build_sparse_graph(
                node_features[0], edge_adj_logits[0], edge_type_logits[0], threshold
            )
            rdkit_mol, smiles = build_rdkit_smiles(active_nodes_cpu, selected_edges, active_edge_types_cpu)
            bond_count = int(pyg_graph.edge_index.size(1) // 2)
            threshold_results.append((threshold, bond_count, int(active_indices.numel()), smiles))
            if best_pack is None and smiles is not None:
                best_pack = (pyg_graph, rdkit_mol, smiles, active_indices, selected_edges, active_nodes_cpu, active_edge_types_cpu)
                best_threshold = threshold

        if best_pack is None:
            pyg_graph, active_indices, selected_edges, active_nodes_cpu, active_edge_types_cpu = build_sparse_graph(
                node_features[0], edge_adj_logits[0], edge_type_logits[0], 0.4
            )
            rdkit_mol, smiles = build_rdkit_smiles(active_nodes_cpu, selected_edges, active_edge_types_cpu, fallback_single_bonds=True)
            best_pack = (pyg_graph, rdkit_mol, smiles, active_indices, selected_edges, active_nodes_cpu, active_edge_types_cpu)
            best_threshold = 0.4

        pyg_graph, rdkit_mol, smiles, active_indices, selected_edges, active_nodes_cpu, active_edge_types_cpu = best_pack
        generated_graphs.append(pyg_graph)
        sample_logs.append({
            'idx': i + 1,
            'thresholds': threshold_results,
            'active_nodes': int(active_indices.numel()),
            'selected_bonds': int(len(selected_edges)),
            'chosen_threshold': best_threshold,
            'smiles': smiles,
        })

        edge_density = float(torch.sigmoid(edge_adj_logits[0]).mean().item())
        passed_filter = passes_quick_halogen_filter(smiles)
        if passed_filter:
            filtered_smiles.append(smiles)
        if (i + 1) % report_every == 0 or passed_filter:
            print(f'  Sample {i+1:4d}: active nodes={int(active_indices.numel())}, selected bonds={len(selected_edges)}, raw edge density={edge_density:.2%}, chosen threshold={best_threshold:.1f}, passes_filter={passed_filter}, smiles={smiles}')
        if (i + 1) % report_every == 0:
            print(f'    Progress: {i+1}/{n_samples} samples processed; passing filter so far={len(filtered_smiles)}')

print(f'\n✓ Generated {len(generated_graphs)} packaged graphs')
print(f'Passed quick filter: {len(filtered_smiles)} / {n_samples}')
print('\nFirst 5 non-chlorine SMILES that passed the quick filter:')
for idx, smiles in enumerate(filtered_smiles[:5], start=1):
    print(f'  {idx}. {smiles}')
if len(filtered_smiles) == 0:
    print('  (No non-chlorine molecules passed the filter.)')

print('\n' + '-'*60)
print('BASELINE: Random Latent Vectors (No Training)')
print('-'*60)

baseline_graphs = []
with torch.no_grad():
    for i in range(baseline_samples):
        z_random = torch.randn(1, latent_dim, device=device)
        try:
            node_features, edge_adj_logits, edge_type_logits = vae.decode(z_random, pocket_embedding=real_pocket)
            pyg_graph, active_indices, selected_edges, active_nodes_cpu, active_edge_types_cpu = build_sparse_graph(
                node_features[0], edge_adj_logits[0], edge_type_logits[0], 0.4
            )
            rdkit_mol, smiles = build_rdkit_smiles(active_nodes_cpu, selected_edges, active_edge_types_cpu)
            baseline_graphs.append(pyg_graph)
            print(f'  Baseline {i+1:2d}: active nodes={int(active_indices.numel())}, bonds={len(selected_edges)}, smiles={smiles}')
        except Exception as e:
            print(f'  Baseline {i+1:2d}: Failed (unexpected decode error)')
            baseline_graphs.append(None)

print('\n' + '='*60)
print('RESULTS & SUMMARY')
print('='*60)

results_dir = ROOT / 'results' / 'generated_mols'
results_dir.mkdir(parents=True, exist_ok=True)

summary = {
    'phase': 'VAE_Generative_Sampling',
    'n_generated': len(generated_graphs),
    'model_path': str(models_dir / 'zinc_pretrained_graphvae.pth'),
    'latent_dim': latent_dim,
    'pocket_dim': pocket_dim,
    'thresholds': thresholds_to_try,
    'node_keep_fraction': node_keep_fraction,
    'atom_temperature': atom_temperature,
    'max_halogens_before_penalty': max_halogens_before_penalty,
    'samples': sample_logs,
}

import json
with open(results_dir / 'vae_generation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nGenerated molecules: {summary["n_generated"]}')
print(f'Model saved to: {summary["model_path"]}')
print(f'Results saved to: {results_dir / "vae_generation_summary.json"}')
print(f'\n✅ VAE generative model complete!')


PHASE 3: Molecule Generation from Learned Latent Space

Generating 1000 molecules by sampling latent space...

✓ Using real pocket embedding with shape: (1, 128)


[17:21:18] non-ring atom 0 marked aromatic
[17:21:18] non-ring atom 0 marked aromatic
[17:21:18] non-ring atom 0 marked aromatic


  Sample  100: active nodes=12, selected bonds=24, raw edge density=0.02%, chosen threshold=0.3, passes_filter=False, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
    Progress: 100/1000 samples processed; passing filter so far=0


[17:21:22] non-ring atom 3 marked aromatic
[17:21:22] Can't kekulize mol.  Unkekulized atoms: 4
[17:21:22] Can't kekulize mol.  Unkekulized atoms: 7


  Sample  200: active nodes=12, selected bonds=24, raw edge density=0.00%, chosen threshold=0.3, passes_filter=False, smiles=ClCl.ClCl.ClCl.ClCl.ClCl.ClCl
    Progress: 200/1000 samples processed; passing filter so far=0


...

  Sample  700: active nodes=12, selected bonds=24, raw edge density=0.00%, chosen threshold=0.3, passes_filter=False, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
    Progress: 700/1000 samples processed; passing filter so far=0
  Sample  800: active nodes=12, selected bonds=24, raw edge density=0.02%, chosen threshold=0.3, passes_filter=False, smiles=ClCl.ClCl.ClCl.ClCl.ClCl.ClCl
    Progress: 800/1000 samples processed; passing filter so far=0


[17:21:41] non-ring atom 7 marked aromatic
[17:21:41] non-ring atom 0 marked aromatic
[17:21:42] non-ring atom 0 marked aromatic
[17:21:42] non-ring atom 0 marked aromatic


  Sample  900: active nodes=12, selected bonds=24, raw edge density=0.01%, chosen threshold=0.3, passes_filter=False, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
    Progress: 900/1000 samples processed; passing filter so far=0


[17:21:44] non-ring atom 0 marked aromatic
[17:21:44] non-ring atom 0 marked aromatic


  Sample 1000: active nodes=12, selected bonds=24, raw edge density=0.00%, chosen threshold=0.3, passes_filter=False, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
    Progress: 1000/1000 samples processed; passing filter so far=0

✓ Generated 1000 packaged graphs
Passed quick filter: 0 / 1000

First 5 non-chlorine SMILES that passed the quick filter:
  (No non-chlorine molecules passed the filter.)

------------------------------------------------------------
BASELINE: Random Latent Vectors (No Training)
------------------------------------------------------------
  Baseline  1: active nodes=12, bonds=24, smiles=ClCl.ClCl.ClCl.ClCl.ClCl.ClCl
  Baseline  2: active nodes=12, bonds=24, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
  Baseline  3: active nodes=12, bonds=24, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
  Baseline  4: active nodes=12, bonds=24, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
  Baseline  5: active nodes=12, bonds=24, smiles=Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
  Baseline  6: active nodes=12, bonds

In [37]:
# ===== PHASE 3 ANALYSIS =====
import json
from collections import Counter

summary_path = ROOT / 'results' / 'generated_mols' / 'vae_generation_summary.json'
with open(summary_path, 'r') as f:
    data = json.load(f)

samples = data.get('samples', [])
smiles_list = [s.get('smiles') for s in samples if s.get('smiles')]
n_total = len(samples)
n_valid_smiles = len(smiles_list)

n_contains_cl = sum('Cl' in s for s in smiles_list)
n_contains_f = sum('F' in s for s in smiles_list)
pass_quick_filter = sum(('Cl' not in s) and (sum(s.count(tok) for tok in ['Cl','Br','F','I']) <= 1) for s in smiles_list)

active_nodes = [int(s.get('active_nodes', 0)) for s in samples]
selected_bonds = [int(s.get('selected_bonds', 0)) for s in samples]
thresholds = [s.get('chosen_threshold') for s in samples]

# Token-level atom heuristic from SMILES strings
atom_counter = Counter()
for s in smiles_list:
    atom_counter['Cl'] += s.count('Cl')
    atom_counter['F'] += s.count('F')
    atom_counter['N'] += s.count('N')
    atom_counter['O'] += s.count('O')
    atom_counter['S'] += s.count('S')
    # crude carbon estimate: uppercase C not part of Cl
    atom_counter['C'] += s.count('C') - s.count('Cl')

print('='*60)
print('PHASE 3 ANALYSIS SUMMARY')
print('='*60)
print(f'Total requested samples: {n_total}')
print(f'Non-empty SMILES: {n_valid_smiles}')
print(f'Quick-filter pass count: {pass_quick_filter} / {n_total} ({(100*pass_quick_filter/max(n_total,1)):.2f}%)')
print(f'SMILES containing Cl: {n_contains_cl} / {n_valid_smiles} ({(100*n_contains_cl/max(n_valid_smiles,1)):.2f}%)')
print(f'SMILES containing F: {n_contains_f} / {n_valid_smiles} ({(100*n_contains_f/max(n_valid_smiles,1)):.2f}%)')

if active_nodes:
    print(f'Active nodes: mean={sum(active_nodes)/len(active_nodes):.2f}, min={min(active_nodes)}, max={max(active_nodes)}')
if selected_bonds:
    print(f'Selected bonds: mean={sum(selected_bonds)/len(selected_bonds):.2f}, min={min(selected_bonds)}, max={max(selected_bonds)}')

threshold_counter = Counter(thresholds)
print(f'Chosen threshold distribution: {dict(threshold_counter)}')
print(f'Approx atom token counts: {dict(atom_counter)}')

print('\nFirst 5 generated SMILES:')
for i, s in enumerate(smiles_list[:5], start=1):
    print(f'  {i}. {s}')


PHASE 3 ANALYSIS SUMMARY
Total requested samples: 1000
Non-empty SMILES: 1000
Quick-filter pass count: 0 / 1000 (0.00%)
SMILES containing Cl: 1000 / 1000 (100.00%)
SMILES containing F: 0 / 1000 (0.00%)
Active nodes: mean=12.00, min=12, max=12
Selected bonds: mean=24.00, min=24, max=24
Chosen threshold distribution: {0.3: 1000}
Approx atom token counts: {'Cl': 11871, 'F': 0, 'N': 1, 'O': 2, 'S': 0, 'C': 126}

First 5 generated SMILES:
  1. Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
  2. Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
  3. Cl.Cl.ClCl.ClCl.ClCl.ClCl.ClCl
  4. Cl.ClC(Cl)(Cl)Cl.ClCl.ClCl.ClCl
  5. ClCl.ClCl.ClCl.ClCl.ClCl.ClCl
